In [ ]:
# =====================================================================
# SYSTEM COMPONENT: IMPORTATION DES LIBRAIRIES ET CONFIGURATION GPU
# =====================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Configuration du matériel (Utilisation du GPU T4 de Colab si disponible)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Appareil utilisé pour l'entraînement : {device}")

# Fixer les seeds pour la reproductibilité
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


# =====================================================================
# PART 1 & 2: SIMULATION/CHARGEMENT ET PRÉPRÉPARATION DU DATASET
# =====================================================================
print("\n--- PART 1 & 2: CHARGEMENT ET PRÉPARATION DES DONNÉES ---")

# Création d'un jeu de données boursières fictif pour s'assurer que le code s'exécute parfaitement
# Remplacez cette section par le chargement de votre fichier réel (ex: pd.read_csv('votre_fichier.csv'))
date_range = pd.date_range(start='2020-01-01', periods=1200, freq='D')
simulated_data = {
    'Open': np.random.uniform(100, 150, size=1200),
    'High': np.random.uniform(150, 200, size=1200),
    'Low': np.random.uniform(50, 100, size=1200),
    'Close': np.sin(np.linspace(0, 50, 1200)) * 25 + 120 + np.random.normal(0, 2, 1200),
    'Volume': np.random.randint(1000, 50000, size=1200),
    'Unnecessary_Col': np.random.rand(1200)
}
df = pd.DataFrame(simulated_data, index=date_range)

# 1. Suppression des colonnes inutiles
df = df.drop(columns=['Unnecessary_Col'])

# 2. Création de la colonne cible (Target) : Prix de clôture du jour suivant (Next Day Close)
df['Target'] = df['Close'].shift(-1)

# Supprimer la dernière ligne qui contient une valeur NaN suite au décalage shift(-1)
df = df.dropna()

print(head_info := df.head())

# 3. Normalisation des caractéristiques (Features) et de la cible (Target)
scaler_features = MinMaxScaler(feature_range=(0, 1))
scaler_target = MinMaxScaler(feature_range=(0, 1))

# Séparation Features / Target
features = df[['Open', 'High', 'Low', 'Close', 'Volume']].values
target = df['Target'].values.reshape(-1, 1)

scaled_features = scaler_features.fit_transform(features)
scaled_target = scaler_target.fit_transform(target)


# =====================================================================
# PART 3: PRÉPARATION DES SÉQUENCES ET SÉPARATION TRAIN/VAL/TEST
# =====================================================================
print("\n--- PART 3: CRÉATION DES SÉQUENCES ET SPLIT ---")

# Fonction pour structurer les données en fenêtres glissantes (ex: 20 jours passés)
def create_sliding_windows(features, target, time_steps=20):
    X, y = [], []
    for i in range(len(features) - time_steps):
        X.append(features[i:(i + time_steps)])
        y.append(target[i + time_steps])
    return np.array(X), np.array(y)

TIME_STEPS = 20
X, y = create_sliding_windows(scaled_features, scaled_target, TIME_STEPS)

# Répartition des indices (80% Train, 10% Validation, 10% Test)
train_end = int(len(X) * 0.8)
val_end = train_end + int(len(X) * 0.1)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

# --- Classe Custom PyTorch Dataset ---
class StockDataset(Dataset):
    def __init__(self, X, y):
        # Conversion explicite en tenseurs PyTorch de type float32
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        
    def __len__(self):
        return len(self.X)
        
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Création des DataLoaders itérables
train_loader = DataLoader(StockDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(StockDataset(X_val, y_val), batch_size=32, shuffle=False)
test_loader = DataLoader(StockDataset(X_test, y_test), batch_size=32, shuffle=False)

print(f"Séquences d'entraînement : {X_train.shape[0]} | Validation : {X_val.shape[0]} | Test : {X_test.shape[0]}")


# =====================================================================
# PART 4: DÉFINITION DE L'ARCHITECTURE DU MODÈLE (LSTM / GRU)
# =====================================================================
print("\n--- PART 4: CONFIGURATION DE L'ARCHITECTURE PYTORCH ---")

class StockLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim=1, dropout_prob=0.2):
        super(StockLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Couche récurrente principale (L'énoncé mentionne l'expérimentation LSTM ou GRU)
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout_prob if num_layers > 1 else 0)
        
        # Couche de régularisation Dropout
        self.dropout = nn.Dropout(dropout_prob)
        
        # Couche Linéaire Dense pour la sortie de régression finale
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # Initialisation des états cachés et de cellule à zéro
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        
        # Propagation avant à travers la couche récurrente
        out, _ = self.lstm(x, (h0, c0))
        
        # Extraction de la sortie du dernier pas de temps (Last Time Step)
        out = self.dropout(out[:, -1, :])
        
        # Passage dans la couche Fully Connected
        out = self.fc(out)
        return out

# Instanciation du modèle avec 5 caractéristiques d'entrée, 64 neurones cachés et 2 couches empilées
model = StockLSTM(input_dim=5, hidden_dim=64, num_layers=2, dropout_prob=0.2).to(device)
print(model)


# =====================================================================
# PART 5: ENTRAÎNEMENT ET VALIDATION DU MODÈLE
# =====================================================================
print("\n--- PART 5: CONFIGURATION ET LANCEMENT DU TRAINING LOOP ---")

# Définition de la fonction de perte (MSE) et de l'optimiseur (Adam)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

EPOCHS = 20
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    # Mode Entraînement
    model.train()
    batch_train_losses = []
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Remise à zéro des gradients calculés au pas précédent
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        # Backward pass et mise à jour des poids
        loss.backward()
        optimizer.step()
        
        batch_train_losses.append(loss.item())
        
    # Mode Évaluation (Validation)
    model.eval()
    batch_val_losses = []
    with torch.no_grad(): # Désactivation du calcul des gradients pour économiser la mémoire vRAM
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            val_loss = criterion(outputs, targets)
            batch_val_losses.append(val_loss.item())
            
    # Stockage et affichage des moyennes de pertes
    epoch_train_loss = np.mean(batch_train_losses)
    epoch_val_loss = np.mean(batch_val_losses)
    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    
    if epoch % 2 == 0 or epoch == 1:
        print(f"Epoch [{epoch}/{EPOCHS}] -> Loss Entraînement : {epoch_train_loss:.5f} | Loss Validation : {epoch_val_loss:.5f}")


# =====================================================================
# PART 6: ÉVALUATION DES PERFORMANCES ($R^2$ SCORE) ET SAUVEGARDE
# =====================================================================
print("\n--- PART 6: ÉVALUATION FINALE SUR LE TEST SET ---")

model.eval()
all_predictions = []
all_targets = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        
        # Transfert vers le CPU pour conversion numpy
        all_predictions.append(outputs.cpu().numpy())
        all_targets.append(targets.numpy())

# Regroupement des listes de batchs
predictions_scaled = np.vstack(all_predictions)
targets_scaled = np.vstack(all_targets)

# Inversion de la normalisation pour obtenir les prix originaux en dollars/valeur réelle
predictions_actual = scaler_target.inverse_transform(predictions_scaled)
targets_actual = scaler_target.inverse_transform(targets_scaled)

# Calcul de la métrique d'évaluation R² Score
r2 = r2_score(targets_actual, predictions_actual)
print(f"🎯 Score de Détermination R² final sur le jeu de test : {r2:.4f}")

# --- Visualisation des résultats de prédiction ---
plt.figure(figsize=(12, 5))
plt.plot(targets_actual, label='Prix Réels (Ground Truth)', color='black', alpha=0.8)
plt.plot(predictions_actual, label='Prédictions LSTM', color='dodgerblue', linestyle='--')
plt.title(f'Prédiction des prix de clôture boursiers (R² = {r2:.4f})')
plt.xlabel('Index de Temps (Jours)')
plt.ylabel('Prix de Clôture')
plt.legend()
plt.tight_layout()
plt.show()

# --- Sauvegarde des artefacts (Modèle et Scaler) ---
import jobfile_or_pickle_placeholder = True
import pickle

# Sauvegarde des poids du modèle PyTorch
torch.save(model.state_dict(), 'pytorch_lstm_stock_model.pth')

# Sauvegarde de l'objet Scaler pour les prédictions futures en production
with open('target_scaler.pkl', 'wb') as f:
    pickle.dump(scaler_target, f)

print("\n✅ Tous les fichiers (`pytorch_lstm_stock_model.pth` et `target_scaler.pkl`) ont été enregistrés avec succès. Prêt pour le push GitHub !")